# Network Anomaly Detection on UNSW-NB15: Random Forest vs XGBoost

Task: binary classification, 0 = normal, 1 = attack.

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from xgboost import XGBClassifier

import pandas as pd
import numpy as np
import json
import joblib

from sklearn.preprocessing import OrdinalEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from xgboost import XGBClassifier


## 2. Load data and combine the two files

In [2]:
train_df = pd.read_csv("UNSW_NB15_training-set.csv", encoding="utf-8-sig")
test_df = pd.read_csv("UNSW_NB15_testing-set.csv", encoding="utf-8-sig")

target = "label"
drop_columns = ["id", "attack_cat", "label"]

X_train = train_df.drop(columns=drop_columns)
y_train = train_df[target]

X_test = test_df.drop(columns=drop_columns)
y_test = test_df[target]

feature_columns = X_train.columns.tolist()

categorical_columns = ["proto", "service", "state"]
numeric_columns = [
    col for col in feature_columns
    if col not in categorical_columns
]

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("Number of features:", len(feature_columns))
print("Target: label (0 = normal, 1 = attack)")

Training shape: (175341, 42)
Testing shape: (82332, 42)
Number of features: 42
Target: label (0 = normal, 1 = attack)


## 3. Preprocess (as in the paper)
Drop `id` and `attack_cat` (it would leak the answer), then label-encode `proto`, `service`, `state`.

In [3]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1
            ),
            categorical_columns
        ),
        ("numeric", "passthrough", numeric_columns)
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

## 4. Models
Settings from the paper: Random Forest with 100 trees; XGBoost with defaults and `logloss`.

Min-Max scaling is placed inside a pipeline, so it is fitted only on the training part of each fold.

In [4]:
rf = make_pipeline(
    preprocessor,
    MinMaxScaler(),
    RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )
)

xgb = make_pipeline(
    preprocessor,
    MinMaxScaler(),
    XGBClassifier(
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )
)

## 5. 5-fold stratified cross-validation

In [5]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

models = {
    "Random Forest": rf,
    "XGBoost": xgb
}

cv_results = {}

for name, model in models.items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=["accuracy", "precision", "recall", "f1", "roc_auc"],
        n_jobs=1
    )

    cv_results[name] = scores["test_f1"].mean()
    print(
        f"{name}: "
        f"F1 = {scores['test_f1'].mean():.3f}, "
        f"Accuracy = {scores['test_accuracy'].mean():.3f}"
    )

Random Forest: F1 = 0.971, Accuracy = 0.960
XGBoost: F1 = 0.970, Accuracy = 0.959


## SECTION 6: Train final model

In [6]:

best_name = max(cv_results, key=cv_results.get)
best_model = models[best_name]

best_model.fit(X_train, y_train)

# Evaluate on the separate testing set
y_pred = best_model.predict(X_test)

print("Selected model:", best_name)
print("Test accuracy:", accuracy_score(y_test, y_pred))
print("Test precision:", precision_score(y_test, y_pred, zero_division=0))
print("Test recall:", recall_score(y_test, y_pred, zero_division=0))
print("Test F1:", f1_score(y_test, y_pred, zero_division=0))

Selected model: Random Forest
Test accuracy: 0.8728076568041587
Test precision: 0.8196522887323944
Test recall: 0.985926056648725
Test F1: 0.8951331864610454


##  SECTION 7: Save model bundle

In [7]:
model_bundle = {
    "model": best_model,
    "feature_columns": feature_columns,
    "target_name": "label",
    "class_names": {
        0: "Normal",
        1: "Attack"
    },
    "selected_model": best_name
}

joblib.dump(model_bundle, "model_bundle.joblib")

# Save one real test record without the target or dropped columns
sample_input = json.loads(
    X_test.iloc[[0]].to_json(orient="records")
)[0]

with open("sample_input.json", "w", encoding="utf-8") as f:
    json.dump(sample_input, f, indent=4)

print("Saved: model_bundle.joblib")
print("Saved: sample_input.json")

Saved: model_bundle.joblib
Saved: sample_input.json


## SECTION 8: Test the saved model

In [9]:
bundle = joblib.load("model_bundle.joblib")

with open("sample_input.json", "r", encoding="utf-8") as f:
    sample = json.load(f)

input_df = pd.DataFrame(
    [sample],
    columns=bundle["feature_columns"]
)

prediction = bundle["model"].predict(input_df)[0]

print("Model:", bundle["selected_model"])
print("Prediction:", int(prediction))
print("Result:", bundle["class_names"][int(prediction)])

Model: Random Forest
Prediction: 1
Result: Attack
